In [9]:
import os
from nwtrace import *
import pandas as pd
import geopandas as gpd

from pathlib import Path


In [10]:
lines = Path('data/more/full_sewers.geojson')
nodes = Path('data/more/all_node_connections.geojson')

lines_gdf = gpd.read_file(lines)
nodes_gdf = gpd.read_file(nodes)

nodes_gdf = nodes_gdf.to_crs(lines_gdf.crs)


In [23]:
multiple = True
upstream_only = False
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROMMH'
downstream_field = 'TOMH'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls

outputname_extra = "allBC_"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

sewershed = NWTrace(
    network=lines_gdf,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

# additional connections
fittings = gpd.read_file("data/more/fitting_connections.geojson")
nodes_up = (fittings[["FACILITYID", "TO_FIXED"]]
            .dropna(subset=["FACILITYID", "TO_FIXED"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'})
            .to_dict(orient="records"))

sewershed.add_upstream_nodes(nodes_up)


node_lookup, seg_lookup = sewershed.get_lookup_tables()
dir_node_lookup, dir_seg_lookup = sewershed.get_directional_lookup_tables()


Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 374 new segment(s).



In [12]:
errors = utils.verify_network_geometry(
    lines=lines_gdf,
    points=nodes_gdf,
    segment_lookup=seg_lookup,
    line_id_field="FACILITYID",
    point_id_field="FACILITYID",
    threshold=100
)
errors

100%|██████████| 169323/169323 [00:01<00:00, 104993.23it/s]

1176 Errors Found
Average Distance 0.14510461119443013 units


[{'node_id': 'MH5212822131',
  'segment_id': 'SL0000-001-5481',
  'error_t': 'missing node',
  'error_msg': 'node MH5212822131 does not exist in dataset',
  'dist': -1,
  'geometry': <MULTILINESTRING ((322146.483 4852349.718, 322151.524 4852358.259, 322152.94...>},
 {'node_id': 'MH5151728331',
  'segment_id': 'SL0000-017-3071',
  'error_t': 'missing node',
  'error_msg': 'node MH5151728331 does not exist in dataset',
  'dist': -1,
  'geometry': <MULTILINESTRING ((328347.352 4851739.48, 328336.356 4851748.668))>},
 {'node_id': 'MH5194926776',
  'segment_id': 'SL0000-027-4231',
  'error_t': 'missing node',
  'error_msg': 'node MH5194926776 does not exist in dataset',
  'dist': -1,
  'geometry': <MULTILINESTRING ((326791.821 4852171.698, 326795.787 4852171.919))>},
 {'node_id': 'MH5008926783',
  'segment_id': 'SL0000-027-9001',
  'error_t': 'missing node',
  'error_msg': 'node MH5008926783 does not exist in dataset',
  'dist': -1,
  'geometry': <MULTILINESTRING ((326799.977 4850308.313, 3

In [13]:
err_df = gpd.GeoDataFrame.from_dict(errors, geometry="geometry")
err_df = err_df.set_crs(lines_gdf.crs)

err_df.to_file("out/errors.gpkg", driver="GPKG", layer="errorsv1")

err_df

,node_id,segment_id,error_t,error_msg,dist,geometry
0,MH5212822131,SL0000-001-5481,missing node,node MH5212822131 does not exist in dataset,-1.0,"MULTILINESTRING ((322146.483 4852349.718, 3221..."
1,MH5151728331,SL0000-017-3071,missing node,node MH5151728331 does not exist in dataset,-1.0,"MULTILINESTRING ((328347.352 4851739.48, 32833..."
2,MH5194926776,SL0000-027-4231,missing node,node MH5194926776 does not exist in dataset,-1.0,"MULTILINESTRING ((326791.821 4852171.698, 3267..."
3,MH5008926783,SL0000-027-9001,missing node,node MH5008926783 does not exist in dataset,-1.0,"MULTILINESTRING ((326799.977 4850308.313, 3267..."
4,MH4983723191-1,SL0000-028-8451,missing node,node MH4983723191-1 does not exist in dataset,-1.0,"MULTILINESTRING ((323108.835 4850029.195, 3231..."
...,...,...,...,...,...,...
1171,None,SL4000-007,missing segment,segment SL4000-007 does not exist in dataset,-1.0,None
1172,None,SL110491,missing segment,segment SL110491 does not exist in dataset,-1.0,None
1173,None,SL1404719,missing segment,segment SL1404719 does not exist in dataset,-1.0,None
1174,None,SL2000-011,missing segment,segment SL2000-011 does not exist in dataset,-1.0,None


In [14]:
cons_err_df = err_df.groupby(["error_t"]).count()
cons_err_df

,node_id,segment_id,error_msg,dist,geometry
error_t,,,,,
missing node,789,789,789,789,789
missing segment,0,374,374,374,0
spatial,13,13,13,13,13


In [15]:
ids_dup = utils.count_duplicates(lines_gdf, "FACILITYID", minimum_count=0)
ids_dup

{'duplicate_count': {'SL0000-000-0091': 1,
  'SL0000-000-0181': 1,
  'SL0000-000-0271': 1,
  'SL0000-000-0361': 1,
  'SL0000-000-0451': 1,
  'SL0000-000-0541': 1,
  'SL0000-000-0631': 1,
  'SL0000-000-0721': 1,
  'SL0000-000-0722': 1,
  'SL0000-000-0811': 1,
  'SL0000-000-0901': 1,
  'SL0000-000-0991': 1,
  'SL0000-000-1081': 1,
  'SL0000-000-1171': 1,
  'SL0000-000-1261': 1,
  'SL0000-000-1351': 1,
  'SL0000-000-1441': 1,
  'SL0000-000-1531': 1,
  'SL0000-000-1621': 1,
  'SL0000-000-1711': 1,
  'SL0000-000-1801': 1,
  'SL0000-000-1891': 1,
  'SL0000-000-1981': 1,
  'SL0000-000-2071': 1,
  'SL0000-000-2161': 1,
  'SL0000-000-3961': 1,
  'SL0000-000-4051': 1,
  'SL0000-000-4141': 1,
  'SL0000-000-4231': 1,
  'SL0000-000-4232': 1,
  'SL0000-000-4321': 1,
  'SL0000-000-4411': 1,
  'SL0000-000-4501': 1,
  'SL0000-000-4591': 1,
  'SL0000-000-4681': 1,
  'SL0000-000-4771': 1,
  'SL0000-000-4861': 1,
  'SL0000-000-4951': 1,
  'SL0000-000-5041': 1,
  'SL0000-000-5131': 1,
  'SL0000-000-5221': 

In [ ]:
id_field = "FACILITYID"

spatial_err_segs = [err["segment_id"] for err in errors if err.get("error_t") == "spatial"]

segs = lines_gdf.loc[lines_gdf[id_field].isin(spatial_err_segs)]

records = []
for _, line in segs.iterrows():

    endpoints = utils.extract_endpoints(line.geometry)
    
    for role, pt in zip(("from", "to"), endpoints):
        records.append({
            "segment_id": line[id_field],
            "role": role,
            "geometry": pt
        })
        
endpoints_gdf = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs=segs.crs
)

endpoints_gdf["geom_buffer"] = endpoints_gdf.buffer(1)

seg_bufs = gpd.GeoDataFrame(
    endpoints_gdf[["segment_id", "role", "geom_buffer", "geometry"]],
    geometry="geom_buffer",
    crs=endpoints_gdf.crs
).rename(columns={"geometry":"endpoint"})

nearby_nodes = gpd.sjoin(nodes_gdf, seg_bufs, predicate="intersects", how="inner")

nearby_nodes["dist"] = nearby_nodes.geometry.distance(
    nearby_nodes["endpoint"]
)

nearby_nodes

,FACILITYID,TO_ASSET_ID,layer,geometry,index_right,segment_id,role,endpoint,dist
620,JP3572118478,None,ssFITTING,POINT Z (318493.927 4835944.012 0),17,SL1426097-1,to,POINT (318493.927 4835944.012),3.372603e-07
877,JP101361,SL110948,ssFITTING,POINT Z (302088.948 4833716.465 0),21,SL110923,to,POINT (302088.948 4833716.465),3.371031e-07
214413,MH5001912133,None,ssMANHOLE,POINT Z (312149.609 4850242.027 0),4,SL4030218,from,POINT (312149.609 4850242.027),3.357016e-07
243855,MH4729417331,None,ssMANHOLE,POINT Z (317346.732 4847516.616 0),23,SL4050884,to,POINT (317346.51 4847517.266),6.868650e-01
297530,MH2010420,None,ssMANHOLE,POINT Z (304915.635 4829264.625 0),12,SL110441,from,POINT (304915.635 4829264.625),3.368792e-07
297531,MH2010413,None,ssMANHOLE,POINT Z (304804.519 4829231 0),13,SL110441,to,POINT (304804.519 4829231),3.374857e-07
300740,MH2893004953,None,ssMANHOLE,POINT Z (304969.545 4829151.995 0),2,SL2008024,from,POINT (304969.545 4829151.995),3.360867e-07
311207,MH2010447,None,ssMANHOLE,POINT Z (304191.672 4841811.502 0),15,SL110440,to,POINT (304191.672 4841811.502),3.379179e-07
311210,MH2010455,None,ssMANHOLE,POINT Z (304220.987 4841714.053 0),10,SL110439,from,POINT (304220.987 4841714.053),3.365144e-07
311211,MH2010456,None,ssMANHOLE,POINT Z (304205.224 4841799.791 0),11,SL110439,to,POINT (304205.224 4841799.791),3.360642e-07


In [27]:
dir_seg_lookup

{'SL9307': {'from': ['MH3848606859'], 'to': ['MH3847906855']},
 'SL9320': {'from': ['MH3847906855'], 'to': ['MH3839806880']},
 'SL1402455': {'from': ['MH4252412459'], 'to': ['CN8395']},
 'SL1402456': {'from': ['MH4253112486'], 'to': ['CN8396']},
 'SL24393': {'from': ['MH3803706077'], 'to': ['MH3802706070']},
 'SL53336': {'from': ['MH3847506018'], 'to': ['MH3848806064']},
 'SL53337': {'from': ['MH3848806064'], 'to': ['MH3848706071']},
 'SL53338': {'from': ['MH3848706071-1'], 'to': ['MH3849506096']},
 'SL50793': {'from': ['MH3759309686'], 'to': ['MH3756909693']},
 'SL50794': {'from': ['MH3756909693'], 'to': ['MH3752209709']},
 'SL9491': {'from': ['MH3845907082'], 'to': ['MH3837107110']},
 'SL9477': {'from': ['MH3805907172'], 'to': ['MH3803407092']},
 'SL9430': {'from': ['MH3801007013'], 'to': ['MH3791007043']},
 'SL9584': {'from': ['MH3806707199'], 'to': ['MH3799207223']},
 'SL21172': {'from': ['MH3725105628'], 'to': ['MH3722205637']},
 'SL7049': {'from': ['MH3772107502'], 'to': ['MH3778

In [30]:
def not_already_connected(row):
    seg = row["segment_id"]
    role = row["role"]
    node = row["FACILITYID"]  # or whatever field
    
    return node not in dir_seg_lookup[seg][role]

nearby_nodes = nearby_nodes[
    nearby_nodes.apply(not_already_connected, axis=1)
]

idx = (
    nearby_nodes
    .groupby(["segment_id", "role"])["dist"]
    .idxmin()
)

best_candidates = nearby_nodes.loc[idx]

best_candidates

,FACILITYID,TO_ASSET_ID,layer,geometry,index_right,segment_id,role,endpoint,dist
311211,MH2010456,None,ssMANHOLE,POINT Z (304205.224 4841799.791 0),11,SL110439,to,POINT (304205.224 4841799.791),3.360642e-07
311211,MH2010456,None,ssMANHOLE,POINT Z (304205.224 4841799.791 0),14,SL110440,from,POINT (304205.224 4841799.791),3.360642e-07
311211,MH2010456,None,ssMANHOLE,POINT Z (304205.224 4841799.791 0),11,SL110439,to,POINT (304205.224 4841799.791),3.360642e-07
311211,MH2010456,None,ssMANHOLE,POINT Z (304205.224 4841799.791 0),14,SL110440,from,POINT (304205.224 4841799.791),3.360642e-07
321101,MH3344202023A,None,ssMANHOLE,POINT Z (302054.792 4833669.434 0),19,SL110922,to,POINT (302054.792 4833669.434),3.380298e-07
321101,MH3344202023A,None,ssMANHOLE,POINT Z (302054.792 4833669.434 0),20,SL110923,from,POINT (302054.792 4833669.434),3.380298e-07
321101,MH3344202023A,None,ssMANHOLE,POINT Z (302054.792 4833669.434 0),19,SL110922,to,POINT (302054.792 4833669.434),3.380298e-07
321101,MH3344202023A,None,ssMANHOLE,POINT Z (302054.792 4833669.434 0),20,SL110923,from,POINT (302054.792 4833669.434),3.380298e-07
333374,MH3010362,None,ssMANHOLE,POINT Z (318489.389 4835958.492 0),16,SL1426097-1,from,POINT (318489.389 4835958.492),3.360418e-07
243855,MH4729417331,None,ssMANHOLE,POINT Z (317346.732 4847516.616 0),23,SL4050884,to,POINT (317346.51 4847517.266),6.868650e-01
